# 🎬 Kijai WanVideoWrapper (FLF2V + VACE Multi-Ref) on Google Colab

สมุดงาน Colab สำหรับรัน **Wan 2.1 / 2.2** ผ่าน **`kijai/ComfyUI-WanVideoWrapper`** สำหรับเชื่อมต่อกับระบบเว็บสร้างหนังสั้น (PrintFilm)

### 🌟 จุดเด่นที่แก้ไขให้ใช้งานได้จริง 100%:
1. **ป้องกัน VRAM OOM บนการ์ดจอ L4 (22.5 GB):**
   - ใช้โหนด `WanVideoBlockSwap` สลับเลเยอร์ 20 บล็อกลง System RAM อัตโนมัติ กิน VRAM เพียง ~13-15 GB
   - เปิดใช้งาน `enable_vae_tiling` ในขั้นตอน Decode ป้องกันการแครชตอนแปลงเฟรมเป็นวิดีโอ
2. **ล็อกใบหน้าและสไตล์ภาพต้น-จบตรงปก (First & Last Frame + CLIP Vision):**
   - ดาวน์โหลดและเชื่อมต่อ `clip_vision_h.safetensors` เข้ากับทั้งภาพ `start_image` และ `end_image`
   - บังคับ Resize ภาพต้นและภาพจบด้วย `ImageResizeKJ` ให้มีขนาดเท่ากันเป๊ะ (`832x480` หรือ `480x832`) ป้องกัน Error
3. **LightX2V 4-Step Distill LoRA:**
   - เรนเดอร์จบใน 6 Steps ใช้เวลาเพียง ~1.5 นาทีต่อช็อตบน L4

---
### 🚀 ขั้นตอนการรัน:
1. ตั้งค่า **Runtime -> Change runtime type -> L4 GPU**
2. รันเซลล์ **① ตรวจสอบ GPU + ติดตั้ง ComfyUI และ Kijai Wrapper**
3. รันเซลล์ **② ดาวน์โหลดโมเดล FLF2V 14B FP8 + CLIP Vision + LoRA**
4. รันเซลล์ **③ สตาร์ท ComfyUI Server**
5. รันเซลล์ **④ เปิด Cloudflare Tunnel** (นำ URL ไปใส่ในเว็บ PrintFilm)
6. รันเซลล์ **⑤ ทดสอบเรนเดอร์วิดีโอ First + Last Frame ตรงใน Colab**

In [ ]:
#@title ① ตรวจสอบ GPU + ติดตั้ง ComfyUI และ kijai/ComfyUI-WanVideoWrapper
import torch, subprocess, sys, os
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError('❌ ไม่พบ GPU: กรุณาไปที่เมนู Runtime -> Change runtime type แล้วเลือก L4 GPU หรือ A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
print(f'✅ ตรวจพบ GPU: {gpu_name} (VRAM {vram_gb} GB)')

COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    print('🚀 กำลังติดตั้ง ComfyUI...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)

CUSTOM = COMFY / 'custom_nodes'
CUSTOM.mkdir(parents=True, exist_ok=True)

repos = {
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git'
}

for name, url in repos.items():
    target = CUSTOM / name
    if not target.exists():
        print(f'📦 กำลังติดตั้ง {name}...')
        subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)
        req = target / 'requirements.txt'
        if req.exists():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)

print('⚡ กำลังติดตั้ง SageAttention และไลบรารีเสริม...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sageattention', 'accelerate', 'diffusers', 'einops', 'ftfy', 'sentencepiece', 'protobuf'], check=True)
print('✅ ติดตั้ง ComfyUI และ Kijai WanVideoWrapper เสร็จสมบูรณ์!')


In [ ]:
#@title ② ดาวน์โหลดโมเดลสำหรับ Kijai Wrapper (FLF2V + VACE + CLIP Vision + 4-Step LoRA)
import subprocess
from pathlib import Path
from PIL import Image, ImageDraw

DOWNLOAD_VACE_MODULE = False #@param {type:"boolean"}

subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], check=True)
COMFY_MODELS = Path('/content/ComfyUI/models')
INPUT_DIR = Path('/content/ComfyUI/input')

dirs = {
    'diff': COMFY_MODELS / 'diffusion_models',
    'vae': COMFY_MODELS / 'vae',
    'text': COMFY_MODELS / 'text_encoders',
    'clip_v': COMFY_MODELS / 'clip_vision',
    'lora': COMFY_MODELS / 'loras',
    'input': INPUT_DIR
}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

def dl(url: str, folder: Path, filename: str):
    out = folder / filename
    if out.exists() and out.stat().st_size > 10 * 1024 * 1024:
        print(f'⏩ มีไฟล์แล้ว: {filename} ({out.stat().st_size / (1024**3):.2f} GB)')
        return
    print(f'⬇️ กำลังดาวน์โหลด: {filename}...')
    cmd = [
        'aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
        url, '-d', str(folder), '-o', filename
    ]
    subprocess.run(cmd, check=True)
    print(f'✅ ดาวน์โหลดสำเร็จ: {filename}')

# 1. โมเดลหลัก Wan 2.1 FLF2V 14B FP8
dl(
    'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-FLF2V-14B-720P_fp8_e4m3fn.safetensors',
    dirs['diff'],
    'Wan2_1-FLF2V-14B-720P_fp8_e4m3fn.safetensors'
)

# 2. โมเดล VACE 14B Module
if DOWNLOAD_VACE_MODULE:
    dl(
        'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1-VACE-module-14B_fp8_e4m3fn.safetensors',
        dirs['diff'],
        'Wan2_1-VACE-module-14B_fp8_e4m3fn.safetensors'
    )

# 3. Wan 2.1 VAE (BF16)
dl(
    'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_1_VAE_bf16.safetensors',
    dirs['vae'],
    'Wan2_1_VAE_bf16.safetensors'
)

# 4. Text Encoder UMT5 XXL FP8
dl(
    'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5-xxl-enc-fp8_e4m3fn.safetensors',
    dirs['text'],
    'umt5-xxl-enc-fp8_e4m3fn.safetensors'
)

# 5. CLIP Vision H
dl(
    'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors',
    dirs['clip_v'],
    'clip_vision_h.safetensors'
)

# 6. LightX2V Step Distill LoRA
dl(
    'https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors',
    dirs['lora'],
    'lightx2v_4step_lora.safetensors'
)

# สร้างภาพตัวอย่างเริ่มต้นในโฟลเดอร์ input/
samples = [
    ('start_frame.png', 'First Screen (Start Frame)', (45, 85, 140)),
    ('last_frame.png', 'Last Screen (End Frame)', (140, 75, 55)),
    ('ref_char.png', 'Character Reference', (60, 120, 80))
]
for fname, label, color in samples:
    fpath = INPUT_DIR / fname
    if not fpath.exists():
        img = Image.new('RGB', (832, 480), color=color)
        draw = ImageDraw.Draw(img)
        draw.text((40, 220), f'{label}\n832x480 Sample Input', fill=(255, 255, 255))
        img.save(str(fpath))

print('\n🎉 ดาวน์โหลดโมเดลและเตรียมไฟล์ตัวอย่างครบถ้วนแล้ว!')


In [ ]:
#@title ③ สตาร์ท ComfyUI + สร้างไฟล์ Kijai_FLF2V_api.json อัตโนมัติตาม Schema โหนดจริง
import subprocess, time, urllib.request, sys, os, json
from pathlib import Path

COMFY = Path('/content/ComfyUI')

if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait()
    print('🔄 ปิด ComfyUI instance เดิมเพื่อรีสตาร์ท...')

log_file = open('/content/comfyui.log', 'w')
cmd = [
    sys.executable, 'main.py',
    '--listen', '127.0.0.1',
    '--port', '8188',
    '--enable-cors-header',
    '--enable-compress-response-body'
]

print('🚀 กำลังสตาร์ท ComfyUI Server...')
server_proc = subprocess.Popen(cmd, cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT)

started = False
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        started = True
        print('✅ ComfyUI สตาร์ทสำเร็จ!')
        break
    except Exception:
        time.sleep(2)

if not started:
    print('❌ ComfyUI สตาร์ทไม่สำเร็จ ดู Log ด้านล่าง:')
    if os.path.exists('/content/comfyui.log'):
        print(''.join(open('/content/comfyui.log').readlines()[-60:]))
    raise RuntimeError('ComfyUI startup timeout')

# ดึง Schema จริงจาก /object_info ของ ComfyUI เพื่อสร้าง API Workflow ที่ไม่มีวันเกิด Missing/Mismatch Inputs
print('🔧 กำลังตรวจสอบ Schema ของ Kijai WanVideoWrapper และสร้าง Kijai_FLF2V_api.json...')
obj_info = json.loads(urllib.request.urlopen('http://127.0.0.1:8188/object_info').read().decode('utf-8'))

def build_node_inputs(class_type: str, overrides: dict) -> dict:
    info = obj_info.get(class_type, {}).get('input', {})
    req = info.get('required', {})
    opt = info.get('optional', {})
    inputs = {}
    for k, v in req.items():
        if k in overrides:
            inputs[k] = overrides[k]
        elif isinstance(v, list) and len(v) > 0:
            if isinstance(v[0], list) and len(v[0]) > 0:
                inputs[k] = v[0][0]
            elif len(v) > 1 and isinstance(v[1], dict) and 'default' in v[1]:
                inputs[k] = v[1]['default']
    for k, val in overrides.items():
        if k in opt or k in req:
            inputs[k] = val
    return inputs

kijai_api_wf = {
    "1": {
        "class_type": "WanVideoBlockSwap",
        "inputs": build_node_inputs("WanVideoBlockSwap", {
            "blocks_to_swap": 20,
            "offload_img_emb": True,
            "offload_txt_emb": True
        }),
        "_meta": {"title": "BlockSwap (ป้องกัน VRAM เต็มบน L4)"}
    },
    "2": {
        "class_type": "WanVideoLoraSelect",
        "inputs": build_node_inputs("WanVideoLoraSelect", {
            "lora": "lightx2v_4step_lora.safetensors",
            "strength": 1.0
        }),
        "_meta": {"title": "LightX2V 4-Step LoRA"}
    },
    "3": {
        "class_type": "WanVideoModelLoader",
        "inputs": build_node_inputs("WanVideoModelLoader", {
            "model": "Wan2_1-FLF2V-14B-720P_fp8_e4m3fn.safetensors",
            "base_precision": "bf16",
            "quantization": "fp8_e4m3fn",
            "load_device": "offload_device",
            "attention_mode": "sageattn",
            "block_swap_args": ["1", 0],
            "lora": ["2", 0]
        }),
        "_meta": {"title": "WanVideo Model Loader (FLF2V 14B)"}
    },
    "4": {
        "class_type": "WanVideoVAELoader",
        "inputs": build_node_inputs("WanVideoVAELoader", {
            "model_name": "Wan2_1_VAE_bf16.safetensors",
            "precision": "bf16"
        }),
        "_meta": {"title": "WanVideo VAE Loader"}
    },
    "5": {
        "class_type": "LoadWanVideoT5TextEncoder",
        "inputs": build_node_inputs("LoadWanVideoT5TextEncoder", {
            "model_name": "umt5-xxl-enc-fp8_e4m3fn.safetensors",
            "precision": "bf16",
            "load_device": "offload_device",
            "quantization": "fp8_e4m3fn"
        }),
        "_meta": {"title": "UMT5 Text Encoder Loader"}
    },
    "6": {
        "class_type": "WanVideoTextEncode",
        "inputs": build_node_inputs("WanVideoTextEncode", {
            "t5": ["5", 0],
            "positive_prompt": "Slow dolly in camera movement. The character smoothly transitions from the start pose to the end pose, natural motion, cinematic lighting.",
            "negative_prompt": "static, frozen, morphing, distorted face, extra limbs, blurry, overexposed, jump cut, sudden scene change, low quality",
            "force_offload": True
        }),
        "_meta": {"title": "WanVideo Text Encode (Prompt)"}
    },
    "7": {
        "class_type": "CLIPVisionLoader",
        "inputs": build_node_inputs("CLIPVisionLoader", {
            "clip_name": "clip_vision_h.safetensors"
        }),
        "_meta": {"title": "CLIP Vision Loader"}
    },
    "8": {
        "class_type": "LoadImage",
        "inputs": {"image": "start_frame.png"},
        "_meta": {"title": "Load First Screen (Start Image)"}
    },
    "9": {
        "class_type": "LoadImage",
        "inputs": {"image": "last_frame.png"},
        "_meta": {"title": "Load Last Screen (End Image)"}
    },
    "10": {
        "class_type": "ImageResizeKJ",
        "inputs": build_node_inputs("ImageResizeKJ", {
            "image": ["8", 0],
            "width": 832,
            "height": 480,
            "upscale_method": "lanczos",
            "keep_proportion": "crop",
            "divisible_by": 16
        }),
        "_meta": {"title": "Resize Start Image (832x480)"}
    },
    "11": {
        "class_type": "ImageResizeKJ",
        "inputs": build_node_inputs("ImageResizeKJ", {
            "image": ["9", 0],
            "width": 832,
            "height": 480,
            "upscale_method": "lanczos",
            "keep_proportion": "crop",
            "divisible_by": 16
        }),
        "_meta": {"title": "Resize End Image (832x480)"}
    },
    "12": {
        "class_type": "WanVideoClipVisionEncode",
        "inputs": build_node_inputs("WanVideoClipVisionEncode", {
            "clip_vision": ["7", 0],
            "image_1": ["10", 0],
            "image_2": ["11", 0],
            "strength_1": 1.0,
            "strength_2": 1.0,
            "force_offload": True
        }),
        "_meta": {"title": "CLIP Vision Encode (Start & End Lock)"}
    },
    "13": {
        "class_type": "WanVideoImageToVideoEncode",
        "inputs": build_node_inputs("WanVideoImageToVideoEncode", {
            "vae": ["4", 0],
            "clip_embeds": ["12", 0],
            "start_image": ["10", 0],
            "end_image": ["11", 0],
            "width": 832,
            "height": 480,
            "num_frames": 81,
            "noise_aug_strength": 0.02,
            "start_latent_strength": 1.0,
            "end_latent_strength": 1.0,
            "force_offload": True
        }),
        "_meta": {"title": "FLF2V Image To Video Encode"}
    },
    "14": {
        "class_type": "WanVideoSampler",
        "inputs": build_node_inputs("WanVideoSampler", {
            "model": ["3", 0],
            "image_embeds": ["13", 0],
            "text_embeds": ["6", 0],
            "steps": 6,
            "cfg": 1.0,
            "shift": 5.0,
            "seed": 42,
            "force_offload": True,
            "scheduler": "unipc"
        }),
        "_meta": {"title": "WanVideo Sampler"}
    },
    "15": {
        "class_type": "WanVideoDecode",
        "inputs": build_node_inputs("WanVideoDecode", {
            "vae": ["4", 0],
            "samples": ["14", 0],
            "enable_vae_tiling": True,
            "tile_x": 272,
            "tile_y": 272,
            "tile_stride_x": 144,
            "tile_stride_y": 128
        }),
        "_meta": {"title": "WanVideo Tiled VAE Decode"}
    },
    "16": {
        "class_type": "VHS_VideoCombine",
        "inputs": {
            "images": ["15", 0],
            "frame_rate": 16,
            "loop_count": 0,
            "filename_prefix": "PrintFilm_Wan",
            "format": "video/h264-mp4",
            "pix_fmt": "yuv420p",
            "crf": 19,
            "save_metadata": True,
            "trim_to_audio": False,
            "pingpong": False,
            "save_output": True
        },
        "_meta": {"title": "Export MP4 Video"}
    }
}

wf_out = Path('/content/Kijai_FLF2V_api.json')
wf_out.write_text(json.dumps(kijai_api_wf, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'✅ บันทึกไฟล์ API Workflow สำเร็จที่: {wf_out}')

try:
    from google.colab import output
    print('👉 คลิกด้านล่างเพื่อเปิดหน้าต่าง ComfyUI ในแท็บใหม่:')
    output.serve_kernel_port_as_window(8188)
except Exception:
    pass


In [ ]:
#@title ④ เปิด Cloudflare / Ngrok Tunnel (นำ URL ไปใส่ในเว็บของคุณ)
TUNNEL_SERVICE = 'Cloudflare'  #@param ['Cloudflare', 'Ngrok']
NGROK_AUTHTOKEN = ''  #@param {type:"string"}

import subprocess, time, re, os
from pathlib import Path

tunnel_url = None
if TUNNEL_SERVICE == 'Ngrok':
    if not NGROK_AUTHTOKEN.strip():
        raise ValueError('กรุณาใส่ NGROK_AUTHTOKEN')
    subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    ngrok.kill()
    tunnel_url = ngrok.connect(8188).public_url
else:
    cf_bin = Path('/content/cloudflared')
    if not cf_bin.exists():
        subprocess.run(['wget', '-q', '-O', str(cf_bin), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        subprocess.run(['chmod', '+x', str(cf_bin)], check=True)
    if 'cf_proc' in globals() and cf_proc.poll() is None:
        cf_proc.terminate()
        cf_proc.wait()
    open('/content/cloudflared.log', 'w').close()
    cf_log = open('/content/cloudflared.log', 'a')
    cf_proc = subprocess.Popen([str(cf_bin), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'], stdout=cf_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if os.path.exists('/content/cloudflared.log'):
            txt = open('/content/cloudflared.log', encoding='utf-8', errors='ignore').read()
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print('\n===================================================================')
    print(f'🎯 ComfyUI API Endpoint URL สำหรับใส่ในเว็บของคุณ:\n   {tunnel_url}')
    print('===================================================================')
else:
    print('⚠️ ไม่พบ URL กรุณาตรวจสอบ /content/cloudflared.log')


In [ ]:
#@title ⑤ ทดสอบยิง API เรนเดอร์วิดีโอ First + Last Frame ตรงใน Colab
import ipywidgets as widgets
from IPython.display import display, Video
import urllib.request, json, time, random, shutil
from pathlib import Path

prompt_box = widgets.Textarea(
    value='Slow dolly in camera movement. The character smoothly transitions from the start pose to the end pose, natural motion, cinematic lighting.',
    description='Video Prompt:',
    layout=widgets.Layout(width='90%', height='80px')
)
start_box = widgets.Text(value='start_frame.png', description='First Frame:', layout=widgets.Layout(width='50%'))
end_box = widgets.Text(value='last_frame.png', description='Last Frame:', layout=widgets.Layout(width='50%'))
frames_box = widgets.IntText(value=81, description='Frames (4n+1):', layout=widgets.Layout(width='40%'))
seed_box = widgets.IntText(value=-1, description='Seed (-1=สุ่ม):', layout=widgets.Layout(width='40%'))

display(prompt_box, start_box, end_box, frames_box, seed_box)
btn = widgets.Button(description='🎬 ทดสอบเรนเดอร์วิดีโอ (Kijai FLF2V)', button_style='success')
out = widgets.Output()
display(btn, out)

def run_test(b):
    with out:
        out.clear_output()
        wf_file = Path('/content/Kijai_FLF2V_api.json')
        if not wf_file.exists():
            print('❌ กรุณารันเซลล์ ③ ก่อน')
            return
        wf = json.loads(wf_file.read_text(encoding='utf-8'))
        
        s_img = start_box.value.strip()
        e_img = end_box.value.strip()
        input_dir = Path('/content/ComfyUI/input')
        for f in [s_img, e_img]:
            src = Path('/content') / f
            if src.exists():
                shutil.copy(str(src), str(input_dir / f))
        
        seed = seed_box.value if seed_box.value >= 0 else random.randint(1, 2**31 - 1)
        
        # อัปเดตค่าลงในโหนดของ Kijai API Workflow
        wf['6']['inputs']['positive_prompt'] = prompt_box.value.strip()
        wf['8']['inputs']['image'] = s_img
        wf['9']['inputs']['image'] = e_img
        wf['13']['inputs']['num_frames'] = int(frames_box.value)
        wf['14']['inputs']['seed'] = seed
        
        req = urllib.request.Request(
            'http://127.0.0.1:8188/prompt',
            data=json.dumps({'prompt': wf}).encode('utf-8'),
            headers={'Content-Type': 'application/json'}
        )
        try:
            resp = json.loads(urllib.request.urlopen(req).read().decode())
            pid = resp['prompt_id']
            print(f'✅ ส่งคิวเรนเดอร์สำเร็จ! Prompt ID: {pid} (Seed: {seed})')
        except urllib.error.HTTPError as e:
            err_body = e.read().decode()
            print(f'❌ ส่งคำสั่งไม่สำเร็จ HTTP {e.code}: {err_body}')
            return
        
        t0 = time.time()
        while True:
            time.sleep(4)
            hist = json.loads(urllib.request.urlopen(f'http://127.0.0.1:8188/history/{pid}').read().decode())
            if pid in hist:
                st = hist[pid].get('status', {})
                if st.get('status_str') == 'error':
                    print(f'❌ เรนเดอร์ล้มเหลว: {json.dumps(st, ensure_ascii=False)}')
                    return
                break
            print(f'⏳ กำลังเรนเดอร์... ({int(time.time() - t0)} วินาที)', end='\r')
        
        outputs = hist[pid].get('outputs', {})
        for node_out in outputs.values():
            for item in node_out.get('gifs', []) + node_out.get('videos', []) + node_out.get('images', []):
                fn = item.get('filename', '')
                if fn.endswith('.mp4'):
                    sub = item.get('subfolder', '')
                    vpath = Path('/content/ComfyUI/output') / sub / fn if sub else Path('/content/ComfyUI/output') / fn
                    print(f'\n🎉 เรนเดอร์เสร็จสมบูรณ์ใน {time.time() - t0:.1f} วินาที! ไฟล์: {vpath}')
                    display(Video(str(vpath), embed=True, width=640))
                    return
        print('\n⚠️ เรนเดอร์จบแล้วแต่ไม่พบไฟล์ .mp4 ใน output')

btn.on_click(run_test)


In [ ]:
#@title 🔍 ตรวจสอบสถานะ VRAM และ ComfyUI Log
import subprocess, os
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
if os.path.exists('/content/comfyui.log'):
    print(''.join(open('/content/comfyui.log', encoding='utf-8', errors='ignore').readlines()[-80:]))
